# Trustworthy Personalised AI — Analysis Dashboard

Research notebook for analysing training data quality, model benchmark results, and conversation samples. All charts are saved to `exports/` as SVG (vector, dissertation-ready) and PNG (high-resolution raster) via plotly + kaleido. Run cells top-to-bottom on first use; individual sections can be re-run independently after that.

In [ ]:
import json
import re
from pathlib import Path
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
from IPython.display import HTML, display

pio.templates.default = "plotly_white"
PALETTE = px.colors.qualitative.Set2

DATA_DIR    = Path("data")
REPORTS_DIR = Path("reports")
EXPORTS_DIR = Path("exports")
EXPORTS_DIR.mkdir(exist_ok=True)

def save_fig(fig, name):
    """Save dissertation-ready SVG + high-res PNG and show inline."""
    fig.write_image(str(EXPORTS_DIR / f"{name}.svg"))
    fig.write_image(str(EXPORTS_DIR / f"{name}.png"), scale=3)
    print(f"\u2713 exports/{name}.svg + .png")
    fig.show()

## Section 1 — Data Loading

In [ ]:
def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def extract_tag_len(content, tag):
    """Return character length of first <tag>\u2026</tag> block, else 0."""
    m = re.search(rf"<{tag}>(.*?)</{tag}>", content, re.DOTALL)
    return len(m.group(1).strip()) if m else 0

SPLITS = {
    "train":             DATA_DIR / "train_sft_v2.jsonl",
    "eval":              DATA_DIR / "eval_sft_v2.jsonl",
    "train_interleaved": DATA_DIR / "train_interleaved.jsonl",
    "train_partB":       DATA_DIR / "train_partB.jsonl",
}

ALL_RECORDS = []   # full records — used by conversation renderer (Section 7)
rows = []

for split_name, path in SPLITS.items():
    if not path.exists():
        continue
    for rec in load_jsonl(path):
        meta = rec.get("metadata", {}).copy()
        msgs = rec.get("messages", [])
        asst = " ".join(m["content"] for m in msgs if m["role"] == "assistant")
        rows.append({
            **meta,
            "split":          split_name,
            "num_messages":   len(msgs),
            "response_chars": len(asst),
            "think_chars":    extract_tag_len(asst, "think"),
            "answer_chars":   extract_tag_len(asst, "answer"),
            "_idx":           len(ALL_RECORDS),
        })
        ALL_RECORDS.append(rec)

df = pd.DataFrame(rows)
print(f"Loaded {len(df):,} records | {df['split'].nunique()} splits")
print(df.groupby("split").size().to_string())